In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import io

PROJECT_ROOT  = Path.cwd().parent
SRC_PATH      = PROJECT_ROOT / "src"
DATA_PATH     = PROJECT_ROOT / "data" / "raw" / "train.parquet"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import duckdb_eda_toolkit as det

con = duckdb.connect(database=":memory:")
con.execute(f"CREATE VIEW train AS SELECT * FROM read_parquet('{DATA_PATH.as_posix()}')")

# Análisis Exploratorio de Datos

:::{admonition} Resumen
:class: info
- El conjunto no presenta datos faltantes, pero la variable objetivo `click` está desbalanceada en proporción aproximada de 5 a 1.
- El análisis de cardinalidad y concentración permitió definir estrategias de codificación para las 21 variables categóricas.
- `device_id` y `device_ip` se representan mediante features de frecuencia en escala logarítmica.
- La tasa de clic por hora presenta un patrón multimodal que no se retiene en agregaciones por día ni por franja.
- Las cinco variables numéricas del modelo no presentan dependencia monotónica entre sí.
:::

## Introducción

El Click-Through Rate (CTR) mide la proporción de impresiones publicitarias que resultan en un clic por parte del usuario. Los sistemas de predicción de clics empleados en búsqueda patrocinada y subasta en tiempo real utilizan esta métrica como insumo. La estimación de la probabilidad de clic permite optimizar la asignación de presupuesto publicitario, priorizar inventario con mayor probabilidad de conversión y personalizar la entrega de anuncios.

El modelado de CTR opera sobre conjuntos de datos de gran escala. El volumen de información generado por las plataformas publicitarias alcanza cientos de millones de impresiones diarias. Las variables que describen cada observación son categóricas y de alta cardinalidad, sin estructura ordinal.

Esta libreta documenta el Análisis Exploratorio de Datos (EDA). Las siguientes secciones abordan verificación de registros faltantes, análisis de la distribución de la variable objetivo, caracterización de la cardinalidad de las variables categóricas y estudio de la estructura temporal de las impresiones.

## Análisis preliminar

El conjunto de datos estuvo compuesto por 40,428,967 registros correspondientes a eventos de impresión publicitaria. El volumen del archivo original exigió una estrategia específica de almacenamiento y procesamiento: los archivos se convirtieron a formato Parquet con compresión Snappy y el procesamiento se ejecutó mediante `DuckDB`.

:::{admonition} Nota: Almacenamiento y Procesamiento
:class: note

La conversión a Parquet con compresión Snappy reorganiza la información en formato columnar. Este formato reduce el tamaño en disco y permite lecturas selectivas por columna. Snappy prioriza la rapidez de descompresión sobre la máxima reducción de tamaño.

`DuckDB` es un motor de consultas columnar que opera directamente sobre archivos Parquet. El conjunto de datos permanece en disco durante el procesamiento. pandas materializa el conjunto completo en la memoria RAM; las consultas de `DuckDB` se ejecutan en streaming. El motor columnar habilita el análisis del conjunto completo mediante consultas eficientes.
:::

Las veinticuatro variables analizadas se distribuyeron en tres tipos generales: identificadores y marcas temporales en formato entero (`INTEGER`), indicadores de baja cardinalidad en formato entero reducido (`TINYINT` y `SMALLINT`) y variables categóricas anonimizadas o de alta cardinalidad en formato de texto (`VARCHAR`). No se observaron datos faltantes en ninguna variable.

In [2]:
_ = det.relation_integrity_table(
    con, "train",
    column_titles=("Variable", "Tipo de dato", "Valores faltantes")
)

Variable,Tipo de dato,Valores faltantes,Variable,Tipo de dato,Valores faltantes
id,INTEGER,0,device_ip,VARCHAR,0
click,TINYINT,0,device_model,VARCHAR,0
hour,INTEGER,0,device_type,TINYINT,0
C1,SMALLINT,0,device_conn_type,TINYINT,0
banner_pos,TINYINT,0,C14,INTEGER,0
site_id,VARCHAR,0,C15,SMALLINT,0
site_domain,VARCHAR,0,C16,SMALLINT,0
site_category,VARCHAR,0,C17,INTEGER,0
app_id,VARCHAR,0,C18,INTEGER,0
app_domain,VARCHAR,0,C19,INTEGER,0


La variable objetivo `click` se codificó de forma binaria y registra la ocurrencia de una interacción por parte del usuario con la impresión publicitaria ($0$ = impresión sin clic; $1$ = impresión con clic). La proporción entre ambas clases resultó cercana a 5 a 1, con las impresiones sin clic representando el 83% del total.

In [3]:
_, _, _ = det.plot_categorical_distribution(
    con, "train",
    column="click",
    level_mapping={0: "No hubo clic (0)", 1: "Hubo clic (1)"},
    title="Distribución de Clics en los Anuncios",
    xlabel="", ylabel="Porcentaje (%)",
    figsize=(5, 4), value_offset=3.0,
    y_step=10, label_size=12, tick_size=11, title_size=13
)

El conjunto de datos se dividió en dos segmentos: entrenamiento y prueba. La asignación de registros se realizó mediante una función de hash determinista sobre el identificador de cada impresión, lo cual garantiza reproducibilidad, independencia del orden original y preservación de la proporción de la variable objetivo en ambos segmentos.

:::{admonition} Nota: Partición Estratificada por Hash
:class: note

La partición asigna cada registro a un segmento mediante el residuo de una función de hash aplicada sobre el identificador único:

$$
\text{segmento}(id) =
\left\{
\begin{array}{ll}
\text{Entrenamiento}, & \text{si } \text{hash}(id) \bmod 100 < 70 \\[0.5em]
\text{Prueba}, & \text{en caso contrario}
\end{array}
\right.
$$

La función de hash distribuye los identificadores de forma uniforme sobre los cien buckets disponibles. Como el hash es independiente del target, la proporción de clics converge a la proporción global en cada segmento con error despreciable para el tamaño del conjunto.
:::

El segmento de entrenamiento incluyó 28,294,671 observaciones (69.99%). El segmento de prueba incluyó 12,134,296 observaciones (30.01%). El EDA se realizó exclusivamente sobre el segmento de entrenamiento; el segmento de prueba permaneció reservado sin procesar.

In [5]:
view_name = det.create_split_column(
    con,
    "train",
    id_column="id",
    train_ratio=0.70,
)

_ = det.split_summary_table(
    con, "train_split",
    target_labels={
        0: "No hubo clic (0)",
        1: "Hubo clic (1)",
    },
    split_labels={
        "Entrenamiento": "Entrenamiento",
        "Prueba": "Prueba",
    },
    total_label="Total"
)

,Entrenamiento,Prueba
No hubo clic (0),"23,490,605 (83.02%)","10,073,296 (83.02%)"
Hubo clic (1),"4,804,066 (16.98%)","2,061,000 (16.98%)"
Total,"28,294,671 (69.99%)","12,134,296 (30.01%)"


El desbalance de la variable objetivo direcciona la selección de métricas de evaluación. La exactitud resulta inadecuada para este problema. Las métricas de F1-score, AUC-ROC y recall ofrecerían una caracterización más robusta del desempeño del modelo.

## Variables categóricas

Las veintiuna variables categóricas del conjunto presentan cardinalidades que abarcan varios órdenes de magnitud, desde 4 hasta más de 5 millones de valores únicos. `device_ip` y `device_id` concentran las cardinalidades más altas, con 5,605,227 y 2,134,228 valores únicos respectivamente. `app_id` ocupa el tercer lugar con 7,958 valores únicos.

`device_ip` representa la dirección IP de la impresión y `device_id` el identificador anónimo del dispositivo. Ninguna de las dos se considera en los análisis posteriores. Cada categoría concentra un número de observaciones insuficiente para estimar una tasa de clic estable, y su codificación generaría columnas con varianza cercana a cero. La información que podrían aportar admite una representación agregada como feature derivada. `app_id` se conserva por su cardinalidad manejable.

In [6]:
CATEGORICAL_COLUMNS = [
    "site_id", "site_domain", "site_category",
    "app_id", "app_domain", "app_category",
    "device_id", "device_ip", "device_model",
    "C1", "C14", "C15", "C16", "C17", "C18", "C19", "C20", "C21",
    "banner_pos", "device_type", "device_conn_type",
]

con.execute("""
    CREATE OR REPLACE VIEW train_set AS
    SELECT * FROM train_split WHERE split = 'Entrenamiento'
""")

_ = det.categorical_cardinality_table(
    con, "train_set",
    columns=CATEGORICAL_COLUMNS,
    column_titles=("Variable", "Valores únicos")
)

Variable,Valores únicos,Variable,Valores únicos
device_ip,"5,605,227",C21,60
device_id,"2,134,228",app_category,34
app_id,"7,958",site_category,26
device_model,"7,931",C16,9
site_domain,"7,085",C15,8
site_id,"4,541",C1,7
C14,"2,606",banner_pos,7
app_domain,514,device_type,5
C17,434,C18,4
C20,171,device_conn_type,4


El contraste de independencia entre cada variable categórica y `click` identifica ocho variables con asociación significativa tras la corrección de Bonferroni. Solo cuatro de ellas superan una V de Cramér de .10, mientras que las restantes presentan asociaciones débiles por debajo de ese umbral. Ninguna variable alcanza un tamaño de efecto moderado-alto.

:::{admonition} Nota: Supuesto de Cochran
:class: note

La aproximación asintótica del estadístico χ² a su distribución teórica requiere que las frecuencias esperadas de la tabla de contingencia cumplan ciertas condiciones. La regla de Cochran establece que ninguna celda debe tener frecuencia esperada inferior a 1 y que no más del 20% de las celdas pueden tener frecuencia esperada inferior a 5. 

Las variables de cardinalidad alta generan un número elevado de celdas con frecuencias esperadas bajas, lo que invalida el contraste.
:::

La capacidad predictiva del modelo dependerá de la combinación de múltiples variables más que de la contribución individual de cada una. `C21` lidera el grupo con el efecto más alto, seguida por `C18`, `C16` y `C15`. Las variables de cardinalidad alta no se incluyen en la tabla porque su tabla de contingencia no satisface el supuesto de Cochran en su formulación estricta.

In [7]:
ASSOCIATION_COLUMNS = [
    "C1", "banner_pos", "site_id", "site_domain", "site_category",
    "app_id", "app_domain", "app_category", "device_model",
    "device_type", "device_conn_type",
    "C14", "C15", "C16", "C17", "C18", "C19", "C20", "C21",
]

_ = det.categorical_association_table(
    con, "train_set",
    columns=ASSOCIATION_COLUMNS, target="click",
    alpha=0.05, p_adjust_method="bonferroni",
    min_expected=1, max_frac_below_5=0.20,
    hide_empty_rows = True
)

Variable,gl,Chi-cuadrado,p-valor ajustado,Cramér's V
C21,59,"1,154,085.617",< .001,.202
C18,3,"851,803.815",< .001,.174
C16,8,"601,516.546",< .001,.146
C15,7,"448,416.732",< .001,.126
device_conn_type,3,"217,855.585",< .001,.088
C1,6,"47,458.821",< .001,.041
device_type,4,"43,549.644",< .001,.039
banner_pos,6,"20,246.858",< .001,.027


`C21` agrupa 60 categorías con una distribución de cola larga. Las cinco categorías más frecuentes acumulan el 56.7% del volumen total. La distribución desciende en escalones: la categoría principal casi duplica a la segunda, y el salto entre la tercera y la cuarta reduce la frecuencia a menos de la mitad. Las 55 categorías restantes reparten el 43.3% del volumen, sin que ninguna de ellas alcance una presencia comparable a las cinco primeras. Los códigos identificadores son anónimos y carecen de interpretación semántica.

In [8]:
n_total = int(con.sql("SELECT COUNT(*) AS n FROM train_set").df()["n"].iloc[0])

_, _, _ = det.plot_categorical_distribution(
    con,
    """
    (SELECT * FROM train_set
     WHERE "C21" IN (
         SELECT "C21" FROM train_set
         GROUP BY "C21"
         ORDER BY COUNT(*) DESC
         LIMIT 5
     ))
    """,
    column="C21", n=n_total,
    title="Distribución de las 5 categorías más frecuentes de C21",
    xlabel="", ylabel="Porcentaje del total (%)",
    figsize=(7, 4), ylim=(0, 50),
    value_offset=1.0,pads=(9, 11, 12),
    y_step=5, label_size=11, tick_size=10, title_size=12
)

`C18` presenta una distribución equilibrada entre sus cuatro categorías, sin concentración extrema en ninguna de ellas. El par de categorías más frecuentes agrupa aproximadamente tres cuartas partes del volumen, y las dos restantes conservan una presencia no marginal. Esta estructura sugiere que la variable conserva capacidad discriminante en todos sus niveles.

In [13]:
n_total = int(con.sql("SELECT COUNT(*) AS n FROM train_set").df()["n"].iloc[0])

subquery_c16_top5 = """
    (SELECT * FROM train_set
     WHERE "C16" IN (
         SELECT "C16" FROM train_set
         GROUP BY "C16" ORDER BY COUNT(*) DESC LIMIT 5
     ))
"""

_, _, _ = det.plot_categorical_distribution_pair(
    con, relation=("train_set", subquery_c16_top5),
    columns=("C18", "C16"), titles=("C18", "C16 (top 5)"),
    xlabels=("", ""), ylabel="Porcentaje del total (%)",
    n=n_total, ylim=(0, 109), y_step=10,
    figsize=(10, 4), tick_size=10, label_size=11, title_size=12
)

`C16` muestra el patrón opuesto. Una única categoría concentra aproximadamente del 95% de las observaciones, mientras que las restantes presentan frecuencias marginales. La variable se aproxima a una constante, y su variabilidad efectiva parece residir en el contraste entre la categoría dominante y el resto.

### Estrategias de codificación

Las variables categóricas con mayor asociación con `click` revelan la necesidad de estudiar una posible re-categorización. `C15`, `C1`, `C16` y `device_type` alcanzan el 90% del volumen con una sola categoría. `device_conn_type` requiere dos, dado que su categoría principal se sitúa en el 86.29% del total. A partir del 75% y del 50%, las cinco se resuelven con una única categoría. La codificación de las categorías no es secuencial: `device_conn_type` usa los valores 0, 2, 3 y 5, y `device_type` usa 0, 1, 2, 4 y 5. La codificación sugerida es un indicador binario que distingue la categoría dominante de las restantes.

In [15]:
DICHOTOMIZE_COLUMNS = [
    "C1", "C16", "C15", "device_type", "device_conn_type"
]

df_cov_dichotomize = det.categorical_coverage_table(
    con, "train_set",
    columns=DICHOTOMIZE_COLUMNS,
    thresholds=(0.99, 0.95, 0.90, 0.75, 0.50),
    column_titles=("Variable", "99%", "95%", "90%", "75%", "50%")
)

Variable,99%,95%,90%,75%,50%
C15,2,2,1,1,1
C1,3,2,1,1,1
C16,3,2,1,1,1
device_type,3,2,1,1,1
device_conn_type,3,3,2,1,1


Cinco variables presentan una distribución intermedia entre la concentración y la cola larga. La categoría dominante cubre entre el 40% y el 72% del volumen, y el 90% se alcanza con dos o tres categorías en `banner_pos`, `C18`, `app_category` y `site_category`. `app_domain` requiere seis. La codificación combina identificadores enteros en `banner_pos` y `C18` con cadenas hash en `app_category`, `site_category` y `app_domain`. Las cinco presentan una cola de categorías con frecuencias marginales. La codificación sugerida consiste en conservar las N categorías más frecuentes y agrupar el resto en una categoría `Other`.

In [16]:
TOPN_COLUMNS = [
    "banner_pos", "app_category", "site_category",
    "app_domain", "C18", "C19", "C20", "C21"
]

df_cov_topn = det.categorical_coverage_table(
    con, "train_set",
    columns=TOPN_COLUMNS,
    thresholds=(0.99, 0.95, 0.90, 0.75, 0.50),
    column_titles=("Variable", "99%", "95%", "90%", "75%", "50%")
)

Variable,99%,95%,90%,75%,50%
banner_pos,2,2,2,2,1
C18,4,4,3,2,2
app_category,5,4,3,2,1
site_category,6,4,3,3,2
app_domain,15,9,6,2,1
C21,40,26,19,10,4
C19,42,28,20,9,2
C20,82,42,23,8,2


Las seis variables restantes requieren entre 36 y 647 categorías para cubrir el 90% del volumen. `app_id` concentra el 50% en una sola categoría pero necesita 620 para el 99%. `site_id` y `site_domain` presentan un patrón similar. `C17`, `C14` y `device_model` no muestran dominante alguna.

:::{admonition} Nota: Target Encoding
:class: note

El target encoding reemplaza cada categoría por la tasa media del target en las observaciones que pertenecen a esa categoría. Para una categoría $c$ con $n_c$ observaciones y tasa de clic $\bar{y}_c$:

$$
\text{TE}(c) = \bar{y}_c
$$

El encoder se ajusta exclusivamente sobre el conjunto de entrenamiento y se aplica sin modificación a los conjuntos de validación y prueba. Las categorías no vistas reciben la tasa global $\bar{y}$ del entrenamiento.

El suavizado hacia la tasa global controla el ruido de las categorías con pocas observaciones:

$$
\text{TE}(c) = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}}{n_c + m}
$$

El parámetro $m$ define cuánta masa debe acumular una categoría para que su tasa propia domine sobre la global. Las categorías con pocas observaciones reciben valores cercanos a $\bar{y}$, lo cual evita que el modelo aprenda pesos a partir de tasas inestables.
:::

La codificación combina dos estrategias. Para `app_id`, `site_id` y `site_domain`, la categoría dominante se conserva como indicador binario y el resto se codifica mediante target encoding con suavizado. Para `C17`, `C14` y `device_model`, el target encoding se aplica sobre la totalidad de las categorías. Las dos últimas presentan una distribución uniforme sobre cientos de categorías y asociación despreciable con el target, por lo que se sugiere excluirlas del modelado.

In [17]:
CONCENTRATION_COLUMNS = [
    "site_id", "site_domain",
    "app_id", "device_model",
    "C14", "C17"
]

df_conc = det.categorical_coverage_table(
    con, "train_set",
    thresholds=(0.99, 0.95, 0.90, 0.75, 0.50),
    column_titles=("Variable", "99%", "95%", "90%", "75%", "50%"),
    columns=CONCENTRATION_COLUMNS
)

Variable,99%,95%,90%,75%,50%
C17,272,171,124,60,22
app_id,620,108,36,6,1
site_domain,651,136,52,12,2
site_id,821,220,86,17,2
C14,1202,624,374,164,53
device_model,2523,1141,647,208,46


La representación agregada de `device_ip` y `device_id` se concreta en dos features numéricas: el número de impresiones asociadas a cada `device_id` y el número de impresiones asociadas a cada `device_ip`.  Ambas features son generalizables: un dispositivo o una dirección nuevos reciben una frecuencia baja independientemente de si su identificador apareció en el conjunto de entrenamiento. La codificación conserva la información de recurrencia sin expandir la dimensionalidad.

`device_id` contiene un valor placeholder (`a99f214a`) que concentra el 82% de las observaciones, correspondiente a impresiones sin identificador de dispositivo. La feature derivada asigna frecuencia cero a las filas del placeholder, de modo que el modelo distingue las impresiones sin identificador de aquellas asociadas a dispositivos recurrentes. No se añade un indicador binario adicional porque la frecuencia cero ya captura esa distinción.

In [18]:
_ = det.show_encoding_strategy_table()

Variable,Estrategia,K,Variable,Estrategia,K
C1,Dicotomizar,—,C21,Top-K + Other,10
C15,Dicotomizar,—,site_category,Top-K + Other,3
C16,Dicotomizar,—,app_id,Híbrida (binaria + TE),—
device_conn_type,Dicotomizar,—,site_domain,Híbrida (binaria + TE),—
device_type,Dicotomizar,—,site_id,Híbrida (binaria + TE),—
app_category,Top-K + Other,3,C14,Target encoding,—
app_domain,Top-K + Other,6,C17,Target encoding,—
banner_pos,Top-K + Other,2,device_model,Target encoding,—
C18,Top-K + Other,3,device_id,Feature derivada,—
C19,Top-K + Other,10,device_ip,Feature derivada,—


## Caracterización temporal

La variable `hour` en formato `YYMMDDHH` se descompuso en tres features temporales: día del mes, componentes cíclicas de la hora y franja horaria. La variable original no se conserva en el modelo.

:::{admonition} Nota: Componentes cíclicas de la hora
:class: note

La hora del día es una variable cíclica. La distancia entre las 23:00 y las 00:00 es de una hora, pero como enteros la diferencia es de 23 unidades. La codificación con seno y coseno proyecta cada hora sobre un círculo unitario:

$$
\text{hour\_sin} = \sin\left(\frac{2\pi h}{24}\right), \qquad \text{hour\_cos} = \cos\left(\frac{2\pi h}{24}\right)
$$

Donde $h$ es la hora en formato 0–23. Las dos componentes identifican cada hora de forma unívoca y preservan la proximidad entre horas contiguas, incluida la transición entre las 23:00 y las 00:00.
:::

La franja horaria agrupa las horas en cuatro niveles: Madrugada, Mañana, Tarde y Noche. Las componentes de año y mes no se extraen: el dataset cubre diez días de octubre de 2014, de modo que ambas serían constantes.

In [19]:
con.execute("""
    CREATE OR REPLACE VIEW temporal_features AS
    SELECT
        "id" AS id,
        "hour" AS hour_raw,
        "click" AS click,
        CAST("hour" / 100 % 100 AS INTEGER) AS day,
        CAST("hour" % 100 AS INTEGER) AS hour_of_day,
        SIN(2 * PI() * CAST("hour" % 100 AS INTEGER) / 24) AS hour_sin,
        COS(2 * PI() * CAST("hour" % 100 AS INTEGER) / 24) AS hour_cos,
        CASE
            WHEN CAST("hour" % 100 AS INTEGER) BETWEEN 0 AND 5 THEN 'Madrugada'
            WHEN CAST("hour" % 100 AS INTEGER) BETWEEN 6 AND 11 THEN 'Mañana'
            WHEN CAST("hour" % 100 AS INTEGER) BETWEEN 12 AND 17 THEN 'Tarde'
            ELSE 'Noche'
        END AS time_band
    FROM train_set
""")

_ = det.show_preview_table(con, "temporal_features")

id,hour_raw,click,day,hour_of_day,hour_sin,hour_cos,time_band
871065379,14102100,0,21,0,0.000000,1.000000,Madrugada
1015674494,14102100,0,21,0,0.000000,1.000000,Madrugada
-721663256,14102100,0,21,0,0.000000,1.000000,Madrugada
-1368576336,14102100,0,21,0,0.000000,1.000000,Madrugada
178008573,14102100,0,21,0,0.000000,1.000000,Madrugada


La tasa de clic varía a lo largo del día dentro de una banda estrecha, entre 15.95% y 18.55%. La curva presenta un patrón multimodal con tres picos: madrugada temprana, arranque de la mañana y mitad de la tarde. Los valles intermedios son menos pronunciados que los picos, de modo que la propensión al clic se mantiene dentro de un rango acotado durante toda la jornada. La forma de la curva resulta compatible con ritmos de comportamiento digital documentados en la literatura de marketing móvil. El pico de la madrugada temprana se alinea con el uso del smartphone en la cama, el de la mañana con el desplazamiento y la primera revisión del día, y el de la tarde con la pausa laboral o el regreso a casa.

In [20]:
_, _, _ = det.plot_ctr_by_hour(
    con, "temporal_features",
    title="Tasa de clic por hora del día", xlabel="Hora del día", ylabel="Tasa de clic (%)",
    ylim=(15, 20), y_step=1, pads=(10, 14, 12)
)

La tasa de clic agregada por día del mes y por franja horaria mantiene valores próximos entre sí. Ninguno de los dos cortes presenta días atípicos, tendencias marcadas ni diferencias pronunciadas entre categorías. Ambos exhiben un perfil aproximadamente uniforme, en contraste con el patrón multimodal que aparece al desagregar por hora. Este contraste sugiere que la variabilidad temporal del clic se concentra en la granularidad horaria y que los cortes más gruesos la atenúan.

In [21]:
_, _, _ = det.plot_temporal_overview(
    con, "temporal_features", metric="ctr",
    band_order=("Madrugada", "Mañana", "Tarde", "Noche"),
    titles=("Tasa de clic por día", "Tasa de clic por franja horaria"),
    xlabels=("", ""), ylabel="Tasa de clic (%)", show_values=False,
    ylim=(10, 25), y_step=2, pads=(9, 14, 12)
)

## Variables continuas

Ambas features presentan distribuciones de cola larga. La mayoría de los dispositivos y direcciones acumula un número reducido de impresiones, mientras que una fracción pequeña concentra volúmenes altos. Las dos distribuciones siguen un patrón de ley de potencia en escala logarítmica.

La cola de `freq_device_ip` se extiende más allá de las 100,000 impresiones, mientras que la de `freq_device_id` alcanza el orden de las 15,000. La diferencia sugiere que las direcciones IP agregan tráfico de múltiples dispositivos bajo una misma red, mientras que los identificadores de dispositivo mantienen una correspondencia más cercana al usuario individual. En `freq_device_id`, el valor placeholder `a99f214a` no aparece en el histograma porque recibe frecuencia cero y se excluye al aplicar escala logarítmica.

In [22]:
con.execute("""
    CREATE OR REPLACE VIEW device_features AS
    WITH raw AS (
        SELECT
            t."id" AS id,
            CASE
                WHEN t."device_id" = 'a99f214a' THEN 0
                ELSE COUNT(*) OVER (PARTITION BY t."device_id")
            END AS freq_device_id,
            COUNT(*) OVER (PARTITION BY t."device_ip") AS freq_device_ip
        FROM train_set AS t
    )
    SELECT
        id,
        freq_device_id,
        freq_device_ip,
        LN(freq_device_id + 1) AS log_freq_device_id,
        LN(freq_device_ip + 1) AS log_freq_device_ip
    FROM raw
""")

_, _, _ = det.plot_distribution_pair(
    con, "device_features",
    columns=("freq_device_id", "freq_device_ip"),
    titles=("Frecuencia de device_id", "Frecuencia de device_ip"),
    xlabels=("Número de impresiones", "Número de impresiones"),
    ylabel="Frecuencia",
    log_x=True, log_y=True,
    bins=60, bar_alpha=1.0
)

Las componentes cíclicas `hour_sin` y `hour_cos` no se incluyen en este análisis. Ambas son transformaciones deterministas de la hora del día y su distribución refleja la forma de las funciones trigonométricas, no un patrón del dataset. La variable `hour_of_day` se analiza en la sección de caracterización temporal mediante gráficos de barras y una curva de tasa de clic por hora.

La matriz de correlación se calcula sobre las cinco variables numéricas que ingresan al modelo: `day`, `hour_sin`, `hour_cos`, `log_freq_device_id` y `log_freq_device_ip`. Las versiones crudas de las frecuencias y la hora del día como entero se excluyen por ser redundantes con sus transformaciones. Se utiliza el coeficiente de Spearman por su robustez ante distribuciones de cola larga. Los coeficientes resultantes son cercanos a cero en todos los pares. El valor absoluto máximo corresponde a `log_freq_device_id` y `log_freq_device_ip` con 0.09, un efecto residual del mecanismo común de agregación por frecuencia. No se identifica multicolinealidad entre las variables numéricas.

In [25]:
con.execute("""
    CREATE OR REPLACE VIEW numeric_features AS
    SELECT
        tf.id AS id,
        tf.day AS day,
        tf.hour_sin AS hour_sin,
        tf.hour_cos AS hour_cos,
        df.log_freq_device_id AS log_freq_device_id,
        df.log_freq_device_ip AS log_freq_device_ip
    FROM temporal_features AS tf
    LEFT JOIN device_features AS df ON tf.id = df.id
""")

fig, ax, corr = det.plot_spearman_heatmap(
    con, "numeric_features",
    title="Matriz de correlación de Spearman", 
    annot_size=10, cbar_shrink = 1
)